# Preperation Plan
Process the categorical values of sex with one hot encoding, dropping one as to avoid multicollinearity, since one of the planned models is linear regression.

# Split
Split the data into 80% train, 20% test. Random split, since the rows are all independent (each is a separate abalone's measurements). 

In [3]:
import pandas as pd
from ucimlrepo import fetch_ucirepo 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
features = abalone.data.features 
targets = abalone.data.targets 

# split the data into training and test
features_train, features_test, targets_train, targets_test = train_test_split(features, targets, random_state=0, test_size=0.2)

# transformer for encoding the sex categorical data
feature_trans = ColumnTransformer(([("Sex", OneHotEncoder(drop='first'), ['Sex'])]))


In [4]:
# baseline model
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score

# create the model
baselineModel = DummyRegressor(strategy="median")
bModel = Pipeline([('preprocessor', feature_trans), ('baseline', baselineModel)])

# train it
bModel.fit(features_train, targets_train)

# test it, record results for table
baseline_targets_pred_test = bModel.predict(features_test)
print(r2_score(targets_test, baseline_targets_pred_test))
print(mean_absolute_error(targets_test, baseline_targets_pred_test))
print(root_mean_squared_error(targets_test, baseline_targets_pred_test))

-0.0025090155394540403
2.4688995215311005
3.2996302533181563


In [7]:
# linear model
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

#create model
param_grid = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0]}
metrics = ["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"]
ridgePipeline = Pipeline([('preprocessor', feature_trans), ('ridge', Ridge())])
ridge_rms = GridSearchCV(ridgePipeline, param_grid, scoring=metrics, refit="neg_root_mean_squared_error")
ridge_mae = GridSearchCV(ridgePipeline, param_grid, scoring=metrics, refit="neg_mean_absolute_error")
ridge_r2 = GridSearchCV(ridgePipeline, param_grid, scoring=metrics, refit="r2")

#train
ridge_rms.fit(features_train, targets_train)
ridge_mae.fit(features_train, targets_train)
ridge_r2.fit(features_train, targets_train)

# look at results
print(ridge_rms.best_params_)
print(ridge_rms.best_score_)

print(ridge_mae.best_params_)
print(ridge_mae.best_score_)

print(ridge_r2.best_params_)
print(ridge_r2.best_score_)

# the models tested on rms/r2 both chose 0.1 as the best parameter, so look at what the mae looks like for 0.1
ridge_targets_pred_test = ridge_rms.predict(features_test)
print(-(mean_absolute_error(targets_test, ridge_targets_pred_test)))

# we see it is pretty similar to the best found mae, so we can move forward with this as the best ridge model LOOK AT THIS
# record the error scores for making a table later
print(r2_score(targets_test, ridge_targets_pred_test))
print(mean_absolute_error(targets_test, ridge_targets_pred_test))
print(root_mean_squared_error(targets_test, ridge_targets_pred_test))

{'ridge__alpha': 0.1}
-2.880694724751658
{'ridge__alpha': 10.0}
-2.083520187219679
{'ridge__alpha': 0.1}
0.19128140984831338
-2.1379467823951805
0.19532904516377292
2.1379467823951805
2.956176063455312


In [5]:
# k nearest neighbours model
# CONSIDER SCALING
from sklearn.neighbors import KNeighborsRegressor

# create the model
param_grid = {'knn__n_neighbors': [3, 5, 10, 15, 20],
              'knn__weights': ['uniform', 'distance'],
              'knn__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
              'knn__metric': ['manhattan', 'euclidean']}
metrics = ["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"]
knnPipeline = Pipeline([('preprocessor', feature_trans), ('knn', KNeighborsRegressor())])
knn_rms = GridSearchCV(knnPipeline, param_grid, scoring=metrics, refit="neg_root_mean_squared_error")
knn_mae = GridSearchCV(knnPipeline, param_grid, scoring=metrics, refit="neg_mean_absolute_error")
knn_r2 = GridSearchCV(knnPipeline, param_grid, scoring=metrics, refit="r2")

#train
knn_rms.fit(features_train, targets_train)
knn_mae.fit(features_train, targets_train)
knn_r2.fit(features_train, targets_train)

# look at results
print(knn_rms.best_params_)
print(knn_rms.best_score_)

print(knn_mae.best_params_)
print(knn_mae.best_score_)

print(knn_r2.best_params_)
print(knn_r2.best_score_)

# all of them have the same best parameters, record the scores for the table
knn_targets_pred_test = knn_rms.predict(features_test)
print(r2_score(targets_test, knn_targets_pred_test))
print(mean_absolute_error(targets_test, knn_targets_pred_test))
print(root_mean_squared_error(targets_test, knn_targets_pred_test))

{'knn__algorithm': 'auto', 'knn__metric': 'manhattan', 'knn__n_neighbors': 20, 'knn__weights': 'uniform'}
-2.9962946553296077
{'knn__algorithm': 'auto', 'knn__metric': 'manhattan', 'knn__n_neighbors': 20, 'knn__weights': 'uniform'}
-2.296200849422232
{'knn__algorithm': 'auto', 'knn__metric': 'manhattan', 'knn__n_neighbors': 20, 'knn__weights': 'uniform'}
0.125045660328743
0.12562341078555217
2.3395334928229663
3.0815583386102516


- a results table comparing the baseline with at least two suitable learned models under the same evaluation protocol
- the main model settings and a brief explanation of any tuning performed
- at least one relevant visualisation, such as a confusion matrix, ROC or precision–recall curve, residual plot or feature-importance plot
- a short error or residual analysis identifying where a model performs poorly
- a draft conclusion stating the current best model, evidence, limitations and next steps

# Results Table

| model    | root mean squared error | mean absolute error | r2 error |
|----------|-------------------------|---------------------|----------|
| baseline |3.2996302533181563       | 2.4688995215311005  |-0.0025090155394540403 |
| ridge    |2.956176063455312        | 2.1379467823951805  |0.19532904516377292 |
| knn      |3.0815583386102516       | 2.3395334928229663  |0.12562341078555217 |